In [ ]:
# Cell 1: Imports
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

np.random.seed(59) # reproducibility

In [ ]:
# Cell 2: Define grid and time parameters
nx, ny = 100, 100
n_nodes = nx * ny
n_features = 6
train_days = 100
test_days = 30

print("Nodes:", n_nodes)

In [ ]:
# Cell 3: Generate spatial grid
x_coords = np.linspace(0, 1, nx)
y_coords = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x_coords, y_coords)

# Flatten to node list
node_positions = np.column_stack([X.ravel(), Y.ravel()])

In [ ]:
# Cell 4: Helper to generate smooth spatiotemporal fields
def generate_field(base, amp, freq_t, freq_x, freq_y, timesteps):
    """
    Create a synthetic field with sinusoidal variation in space and time.
    """
    t = np.arange(timesteps)
    field = np.zeros((timesteps, ny, nx))
    for i in range(timesteps):
        field[i] = (
            base
            + amp * np.sin(2 * np.pi * (freq_t * t[i] / timesteps
                                        + freq_x * X
                                        + freq_y * Y))
        )
    return field

In [ ]:
# Cell 5: Generate predictor fields (plausible ranges)
# +1 extra timestep so the last test input always has a target label
timesteps_total = train_days + test_days + 1

u_wind   = generate_field(5,  3,  2, 1, 1, timesteps_total)   # m/s
v_wind   = generate_field(2,  2,  3, 1, 2, timesteps_total)   # m/s
u_water  = generate_field(0.5, 0.3, 4, 2, 1, timesteps_total) # m/s
v_water  = generate_field(0.2, 0.2, 5, 1, 2, timesteps_total) # m/s
bed_shear= generate_field(0.1, 0.05, 6, 2, 2, timesteps_total) # N/m^2

# Soil particle size is static in time (random spatial variation)
d50 = 0.2 + 0.05 * np.random.rand(ny, nx)  # mm
d50 = np.broadcast_to(d50, (timesteps_total, ny, nx))

In [ ]:
# Cell 6: Stack features into one array
features = np.stack([u_wind, v_wind, u_water, v_water, bed_shear, d50], axis=-1)
print("Features shape:", features.shape)  # (time, ny, nx, features)

In [ ]:
# Cell 7: Generate labels (bed elevation field)
# Start with a sloping plane + sinusoidal perturbations
bed0 = 0 - 2 * X + 1 * np.sin(2*np.pi*Y)

labels = np.zeros((timesteps_total, ny, nx))
labels[0] = bed0

for t in range(1, timesteps_total):
    # Small erosion/accretion depending on water velocity & shear
    dz = -0.1 * u_water[t] * bed_shear[t] + 0.1 * v_water[t]
    labels[t] = labels[t-1] + dz

In [ ]:
# Cell 8: Convention Alignment
# Flatten (T, ny, nx, F) grid tensors → (T, N, F) node tensors, matching the format produced by d3d_grid_processing.py.

def build_edge_index(mask: np.ndarray):
    """4-connected edges for all valid cells (inlined from d3d_grid_processing)."""
    mmax, nmax = mask.shape
    node_id = -np.ones_like(mask, dtype=int)
    coords = np.argwhere(mask)
    node_id[mask] = np.arange(len(coords))
    src, dst = [], []
    for i, j in coords:
        u = node_id[i, j]
        for di, dj in [(1,0), (-1,0), (0,1), (0,-1)]:
            ii, jj = i + di, j + dj
            if 0 <= ii < mmax and 0 <= jj < nmax and mask[ii, jj]:
                src.append(u)
                dst.append(node_id[ii, jj])
    return np.vstack([src, dst]), coords

mask = np.ones((ny, nx), dtype=bool)   # all cells valid for synthetic data
edge_index_np, node_index_np = build_edge_index(mask)
edge_index = torch.tensor(edge_index_np, dtype=torch.long)

N = ny * nx
X_all = features.reshape(timesteps_total, N, n_features).astype(np.float32)  # (T, N, F)
Y_all = labels.reshape(timesteps_total, N).astype(np.float32)                  # (T, N)

# X[t] predicts Y[t+1]; timesteps_total = train + test + 1 guarantees no OOB
train_X = torch.from_numpy(X_all[:train_days])                                    # (100, N, F)
train_y = torch.from_numpy(Y_all[1 : train_days + 1])                            # (100, N)

test_X  = torch.from_numpy(X_all[train_days : train_days + test_days])           # ( 30, N, F)
test_y  = torch.from_numpy(Y_all[train_days + 1 : train_days + test_days + 1])  # ( 30, N)

print(f"train_X: {train_X.shape}  train_y: {train_y.shape}")
print(f"test_X:  {test_X.shape}   test_y:  {test_y.shape}")
print(f"Nodes: {N}  Edges: {edge_index.shape[1]}")

In [ ]:
# Cell #9: Animation of training data
# Convert to numpy for plotting
train_beds = train_y.numpy().reshape(-1, ny, nx)  # (T, ny, nx) for visualization

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(train_beds[0], cmap="terrain", origin="lower", animated=True)
ax.set_title("Training Data - Day 0")

def update(frame):
    im.set_array(train_beds[frame])
    ax.set_title(f"Training Data - Day {frame}")
    return [im]

ani_train = animation.FuncAnimation(
    fig, update, frames=train_beds.shape[0], interval=300, blit=True, repeat=False
)

plt.close()  # avoid duplicate static plot

# Display animation inline
HTML(ani_train.to_jshtml())

In [ ]:
# For now, use ground truth as "predictions" (replace with model outputs later)
pred_y = test_y.numpy()  # shape (time, N)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(pred_y[0].reshape(ny, nx), cmap="terrain", origin="lower",
               vmin=pred_y.min(), vmax=pred_y.max(), animated=True)

# Add colorbar legend
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Bed Elevation (m)", rotation=270, labelpad=15)

ax.set_title("Day 0")

def update(frame):
    im.set_array(pred_y[frame].reshape(ny, nx))
    ax.set_title(f"Day {frame}")
    return [im]

ani = animation.FuncAnimation(
    fig, update, frames=pred_y.shape[0], interval=500, blit=True, repeat=False
)

plt.close()  # Prevent duplicate static plot

HTML(ani.to_jshtml())

ani.save("test_bed_elevations.mp4", writer="ffmpeg", fps=2)

In [ ]:
# Static snapshot comparison: Day 0 vs. Day 30
# WHY ARE THEY THE SAME?

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

im1 = axes[0].imshow(pred_y[0].reshape(ny, nx), cmap="terrain", origin="lower")
axes[0].set_title("Predicted Bed Elevation - Day 0")
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(pred_y[-1].reshape(ny, nx), cmap="terrain", origin="lower")
axes[1].set_title(f"Predicted Bed Elevation - Day {pred_y.shape[0] - 1}")
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()